<a href="https://colab.research.google.com/github/jana-nf/pbl-mlops-hands-on-lab/blob/main/pbl_mlops_hands_on_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MLOPS-LAB0_INDT

OBJETIVO: Demonstrar o ciclo de vida completo de MLOps

FLUXO: Dados -> Treinamento -> Avaliação -> MLflow Tracking -> Versionamento -> Quality Gate -> Model Registry -> Governança -> Monitoramento -> Data Drift -> Continuous Training -> Pipeline Result

#1. Verificação do ambiente GPU

In [1]:
!nvidia-smi

Tue Sep 22 17:25:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#2. Instalação

In [2]:
!pip install -q mlflow scikit-learn joblib pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.6/144.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#3. Importação das bibliotecas

In [3]:
import os
import json
import hashlib
import shutil
import joblib
import mlflow
import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

#4. Carregamento, Divisão e Exportação dos Dados em CSV

In [4]:
# Carrega o dataset de Cancro de Mama
data = load_breast_cancer()
X, y = data.data, data.target

# Salva uma cópia local em formato CSV para exportação/consulta
os.makedirs("data", exist_ok=True)
df_cancer = pd.DataFrame(X, columns=data.feature_names)
df_cancer['target'] = y
df_cancer.to_csv("data/breast_cancer.csv", index=False)
print("Dataset salvo em: data/breast_cancer.csv")

# Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

Dataset salvo em: data/breast_cancer.csv


#5. Criação, Treinamento e Avaliação do Modelo (ML)

In [5]:
# Cria o modelo
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Treina o modelo
model.fit(X_train, y_train)

# Predição
predictions = model.predict(X_test)

# Métricas
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print("Accuracy:", round(accuracy, 4))
print("F1-Score:", round(f1, 4))

Accuracy: 0.958
F1-Score: 0.967


#6. MLflow Tracking (Rastreamento de Experimentos)

In [8]:
# Define o nome do experimento
mlflow.set_experiment("pbl-mlops-hands-on-lab")

# Inicia a execução do MLflow
with mlflow.start_run() as run:
    # Registra hiperparâmetros
    mlflow.log_param("algorithm", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)

    # Registra métricas de desempenho
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)

    # Captura a ID do run
    run_id = run.info.run_id

print("RUN ID:", run_id)

RUN ID: 3aeca258de09452bbb9fad6377990f18


#7. Versionamento do Artefato e Verificação de Integridade (SHA-256)

In [9]:
# Cria a pasta de destino
os.makedirs("models", exist_ok=True)

MODEL_VERSION = "v1"
model_path = f"models/model_{MODEL_VERSION}.pkl"

# Salva o artefato do modelo no disco
joblib.dump(
    model,
    model_path
)

print("Modelo criado:")
print(model_path)


def sha256_file(filename):
    sha256 = hashlib.sha256()

# Lê o arquivo em blocos binários para evitar estouro de memória
    with open(filename, "rb") as file:
        for block in iter(lambda: file.read(4096), b""):
            sha256.update(block)

    return sha256.hexdigest()

# Calcula e exibe o hash do modelo exportado
model_hash = sha256_file(model_path)

print("Modelo:", model_path)
print("SHA-256:", model_hash)

Modelo criado:
models/model_v1.pkl
Modelo: models/model_v1.pkl
SHA-256: 5911d92693f71afafbd54a16834a822b1fb49cb9049535444241fb1770e3d02e


#8. Quality Gate (Validação Automatizada de Critérios)

In [10]:
MIN_ACCURACY = 0.80
MIN_F1 = 0.80

quality_gate = (
    accuracy >= MIN_ACCURACY
    and
    f1 >= MIN_F1
)

print("Accuracy:", round(accuracy, 4))
print("F1:", round(f1, 4))

if quality_gate:
    print("QUALITY GATE: PASS")
else:
    print("QUALITY GATE: FAIL")

Accuracy: 0.958
F1: 0.967
QUALITY GATE: PASS


#9. Model Registry e Promoção Controlada

In [11]:
os.makedirs("registry", exist_ok=True)

if quality_gate:
    registry_path = f"registry/model_{MODEL_VERSION}.pkl"
    shutil.copy(
        model_path,
        registry_path
    )
    print("PROMOÇÃO: APROVADA")
    print("MODEL REGISTRY:", registry_path)
else:
    print("PROMOÇÃO: BLOQUEADA")

PROMOÇÃO: APROVADA
MODEL REGISTRY: registry/model_v1.pkl


#10. Governança e Metadados do Modelo

In [12]:
metadata = {
    "model_version": MODEL_VERSION,
    "algorithm": "RandomForestClassifier",
    "accuracy": round(accuracy, 4),
    "f1_score": round(f1, 4),
    "sha256": model_hash,
    "mlflow_run_id": run_id,
    "created_at": datetime.now().isoformat()
}

with open(
    "registry/model_metadata.json",
    "w"
) as file:
    json.dump(
        metadata,
        file,
        indent=4
    )

print(json.dumps(metadata, indent=4))

{
    "model_version": "v1",
    "algorithm": "RandomForestClassifier",
    "accuracy": 0.958,
    "f1_score": 0.967,
    "sha256": "5911d92693f71afafbd54a16834a822b1fb49cb9049535444241fb1770e3d02e",
    "mlflow_run_id": "3aeca258de09452bbb9fad6377990f18",
    "created_at": "2026-09-22T17:53:05.758661"
}


#11. Monitoramento de Produção e Simulação de Drift

In [13]:
baseline_accuracy = accuracy

print(
    "Baseline Accuracy:",
    round(baseline_accuracy, 4)
)

# Simulação de alteração nos dados de produção (distorção dos atributos)
X_production = X_test * 1.25

production_predictions = model.predict(
    X_production
)

production_accuracy = accuracy_score(
    y_test,
    production_predictions
)

print(
    "Accuracy baseline:",
    round(baseline_accuracy, 4)
)

print(
    "Accuracy produção:",
    round(production_accuracy, 4)
)

Baseline Accuracy: 0.958
Accuracy baseline: 0.958
Accuracy produção: 0.7273


#12. Detecção de Degradação de Performance

In [14]:
DRIFT_THRESHOLD = 0.10

performance_drop = (
    baseline_accuracy - production_accuracy
)

print(
    "Degradação:",
    round(performance_drop, 4)
)

if performance_drop > DRIFT_THRESHOLD:
    drift_detected = True
    print("ALERTA: DRIFT DETECTADO")
else:
    drift_detected = False
    print("MODELO ESTÁVEL")

Degradação: 0.2308
ALERTA: DRIFT DETECTADO


#13. Treinamento Contínuo (Continuous Training)

In [15]:
if drift_detected:
    print("CONTINUOUS TRAINING: TRIGGERED")

    model_v2 = RandomForestClassifier(
        n_estimators=150,
        random_state=42
    )

    model_v2.fit(
        X_train,
        y_train
    )

    predictions_v2 = model_v2.predict(
        X_test
    )

    accuracy_v2 = accuracy_score(
        y_test,
        predictions_v2
    )

    print("Nova versão: v2")
    print(
        "Accuracy v2:",
        round(accuracy_v2, 4)
    )

CONTINUOUS TRAINING: TRIGGERED
Nova versão: v2
Accuracy v2: 0.958


#14. Exibição do Resultado Final da Pipeline MLOps

In [18]:
print("""
==================================================
           PIPELINE MLOPS - RESULTADO FINAL
==================================================

DADOS
  │
  ▼
TREINAMENTO
  │
  ▼
AVALIAÇÃO
  │
  ▼
MLFLOW TRACKING
  │
  ▼
VERSIONAMENTO
  │
  ▼
QUALITY GATE
  │
  ▼
MODEL REGISTRY
  │
  ▼
MONITORAMENTO
  │
  ▼
DRIFT
  │
  ▼
CONTINUOUS TRAINING

==================================================
""")


           PIPELINE MLOPS - RESULTADO FINAL

DADOS
  │
  ▼
TREINAMENTO
  │
  ▼
AVALIAÇÃO
  │
  ▼
MLFLOW TRACKING
  │
  ▼
VERSIONAMENTO
  │
  ▼
QUALITY GATE
  │
  ▼
MODEL REGISTRY
  │
  ▼
MONITORAMENTO
  │
  ▼
DRIFT
  │
  ▼
CONTINUOUS TRAINING


